In [3]:
import numpy as np
import pandas as pd
import h5py

In [6]:
objs = load_s82_from_hdf5('data/may3_lc.h5')
len(objs)

38825

In [34]:
import numpy as np
import pandas as pd
from scipy.stats import median_abs_deviation as mad
from multiprocessing import Pool, cpu_count

def reject_outliers_moving_window(times, mags, mag_errs, window_size=6, sigma_thresh=2.5):
    mags = np.array(mags)
    times = np.array(times)
    mag_errs = np.array(mag_errs)

    if len(mags) < 2 * window_size + 1:
        return mags, mag_errs

    mask = np.ones(len(mags), dtype=bool)

    for i in range(len(mags)):
        if i < window_size or i >= len(mags) - window_size:
            continue
        window = mags[i - window_size:i + window_size + 1]
        window_mad = mad(window, nan_policy='omit')
        window_mean = np.nanmean(window)

        if window_mad == 0:
            continue

        if np.abs(mags[i] - window_mean) > sigma_thresh * window_mad:
            mask[i] = False

    return mags[mask], mag_errs[mask]

def robust_normalized_excess_variance(args):
    obj_id, lc_data, bands, min_points, window_size, sigma_thresh = args
    band_nxvs = []
    total_points = 0

    for band in bands:
        mags = lc_data['mags'].get(band, [])
        mag_errs = lc_data['magerrs'].get(band, [])
        times = lc_data['times']

        if len(mags) >= min_points:
            mags_filtered, mag_errs_filtered = reject_outliers_moving_window(
                times, mags, mag_errs, window_size, sigma_thresh)

            valid = (mags_filtered < 30.0) & (mag_errs_filtered > 0) & (mag_errs_filtered < 2.0)
            mags_filtered = mags_filtered[valid]
            mag_errs_filtered = mag_errs_filtered[valid]

            N = len(mags_filtered)
            if N < min_points:
                continue

            weights = 1 / mag_errs_filtered**2
            weighted_mean_mag = np.average(mags_filtered, weights=weights)
            weighted_variance = np.average((mags_filtered - weighted_mean_mag)**2, weights=weights)
            mean_err_sq = np.mean(mag_errs_filtered**2)

            sigma_nxv = (weighted_variance - mean_err_sq) / weighted_mean_mag**2

            if sigma_nxv > 0:
                band_nxvs.append(sigma_nxv)
                total_points += N

    if band_nxvs and total_points > 0:
        avg_nxv = np.mean(band_nxvs)
        combined_score = avg_nxv * np.log10(total_points)
    else:
        avg_nxv = -np.inf
        combined_score = -np.inf

    return (obj_id, avg_nxv, total_points, combined_score)

def rank_multiband_light_curves_parallel(light_curves, bands=['u','g','r','i','z'], 
                                         min_points=5, window_size=6, sigma_thresh=2.5,
                                         n_processes=None):
    if n_processes is None:
        n_processes = cpu_count()

    args_list = [
        (lc_data['object_id'], lc_data, bands, min_points, window_size, sigma_thresh)
        for lc_data in light_curves
    ]

    with Pool(n_processes) as pool:
        results = pool.map(robust_normalized_excess_variance, args_list)

    ranking_df = pd.DataFrame(results, columns=[
        'object_id',
        'robust_avg_nxv',
        'total_observations',
        'combined_score'
    ])

    ranking_df.sort_values('combined_score', ascending=False, inplace=True)

    return ranking_df.reset_index(drop=True)




In [60]:
df = rank_multiband_light_curves_parallel(objs, bands=['u', 'g', 'r', 'i', 'z'])


In [64]:
d = df[df['total_observations'] > 800]
d

,object_id,robust_avg_nxv,total_observations,combined_score
28,1459958,0.001931,986,0.005780
66,1463119,0.000966,963,0.002882
78,1427359,0.000903,811,0.002627
88,1424290,0.000785,973,0.002344
108,1465974,0.000661,1060,0.001999
...,...,...,...,...
35723,1451585,0.000003,1612,0.000010
36016,1401309,0.000003,805,0.000009
36391,1385141,0.000002,1089,0.000006
36473,1464956,0.000002,1317,0.000005


In [65]:
d.to_csv('data/s82_lc_ranking.csv', index=False)